In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

**Topology discovery plots**

In [12]:
df = pd.read_csv("results_topology_discovery_123.csv")
df = df[np.isfinite(df["cost_per_obs"])]  # safety filter

# Normalise by total samples per subplot cell (both classes share same denominator)
df["weight"] = 1.0 / df.groupby(["sigma_pos", "sigma_theta", "sigma_encoder", "k"])["cost_per_obs"].transform("size")

encoder_vals = sorted(df["sigma_encoder"].unique())          # columns (x-axis)
pos_theta_vals = sorted(df.groupby(["sigma_pos","sigma_theta"]).groups)  # rows (y-axis)
k_vals = sorted(df["k"].unique()) 

for k, df_k in df.groupby("k"):
    
    fig, axes = plt.subplots(
        len(pos_theta_vals), 
        len(encoder_vals),
        figsize=(4 * len(encoder_vals), 3.5 * len(pos_theta_vals)),
        sharey=False
    )
    axes = np.atleast_2d(axes)  # Safe grid indexing for 1x1 or 1xN shapes

    palette = {True: "#1f77b4", False: "#ff7f0e"}

    for r, (sp, st) in enumerate(pos_theta_vals):
        for c, se in enumerate(encoder_vals):
            ax = axes[r, c]
            
            # Fast filter on the pre-sliced df_k
            sub = df_k[
                (df_k["sigma_pos"] == sp) & 
                (df_k["sigma_theta"] == st) & 
                (df_k["sigma_encoder"] == se)
            ]

            if sub.empty:
                ax.set_visible(False)
                continue

            sns.histplot(
                data=sub, 
                x="cost_per_obs", 
                hue="connected",
                weights="weight", 
                bins=100, 
                ax=ax,
                element="step", 
                fill=True, 
                alpha=0.4,
                palette=palette, 
                legend=(r == 0 and c == 0)
            )

            if r == 0:
                ax.set_title(f"sigma_enc = {se}", fontsize=10)
            ax.set_ylabel(f"sigma_pos={sp}, sigma_θ={st}\nproportion" if c == 0 else "")
            ax.set_xlabel("cost_per_obs" if r == len(pos_theta_vals) - 1 else "")

    # Include k in title and filename
    fig.suptitle(f"cost_per_obs distribution — k = {k} (connected vs not connected)", y=1.01)
    plt.tight_layout()
    plt.savefig(f"topology_cost_grid_k_{k}.png", dpi=150, bbox_inches="tight")
    plt.close(fig)  # Prevents memory accumulation when generating multiple figures

In [8]:
# 1. Filter and get top 10 per group
result = (
    df[df["connected"] == False]
    .sort_values("cost_per_obs")
    .groupby(["sigma_pos", "sigma_theta", "sigma_encoder"], as_index=False)
    .head(10)
)

# 2. Iterate and print
cols = ["sigma_pos", "sigma_theta", "sigma_encoder", "limb_id", "child_id", "cost_per_obs"]

for group_key, group_df in result.groupby(["sigma_pos", "sigma_theta", "sigma_encoder"]):
    print(f"\n=== Group Key: {group_key} ===")
    print(group_df[cols].to_string(index=False))


=== Group Key: (0.001, 0.0017, 0.0009) ===
 sigma_pos  sigma_theta  sigma_encoder  limb_id  child_id  cost_per_obs
     0.001       0.0017         0.0009        0         7    369.255929
     0.001       0.0017         0.0009        6         1    374.976698
     0.001       0.0017         0.0009        0         7    453.658030
     0.001       0.0017         0.0009        0         7    453.658030
     0.001       0.0017         0.0009        0         7    453.658030
     0.001       0.0017         0.0009        0         7    453.658030
     0.001       0.0017         0.0009        0         7    453.658030
     0.001       0.0017         0.0009        0         7    453.658030
     0.001       0.0017         0.0009        0         7    453.658030
     0.001       0.0017         0.0009        0         7    453.658030

=== Group Key: (0.001, 0.0017, 0.009) ===
 sigma_pos  sigma_theta  sigma_encoder  limb_id  child_id  cost_per_obs
     0.001       0.0017          0.009        6  